<div style="background:linear-gradient(135deg,#1e3a8a 0%,#0c1a3d 100%);border-radius:16px;padding:30px 34px;color:#eff6ff;margin-bottom:6px;">
<div style="font-size:12.5px;letter-spacing:3px;text-transform:uppercase;opacity:.65;font-weight:600;">News Article Topic Classification</div>
<div style="font-size:30px;font-weight:800;margin:6px 0 10px;">02 · Preprocessing</div>
<div style="font-size:14.5px;opacity:.92;max-width:640px;line-height:1.55;">Cleans the raw tables, runs targeted EDA, engineers a much richer feature set, and vectorizes everything into GPU-friendly dense matrices.</div>
</div>

<div style="margin:12px 0 6px;">
<span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">01 Load Data</span><span style="display:inline-block;background:#1e3a8a;color:#fff;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;font-weight:600;">02 Preprocessing</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">03 Modeling</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">04 Evaluation & Testing</span><span style="display:inline-block;background:#e2e8f0;color:#64748b;padding:4px 12px;border-radius:20px;font-size:11.5px;margin-right:5px;">05 Visualization</span>
</div>

<div style="margin:8px 0 14px;">
<span style="display:inline-block;background:#eff6ff;border:1px solid #93c5fd;border-radius:8px;padding:7px 14px;font-size:13px;margin-right:8px;">
<b style="color:#1e3a8a;">⬅ Requires</b>&nbsp; <code>artifacts/01_load_data.pkl</code> — run <code>01_load_data.ipynb</code> once beforehand
</span><span style="display:inline-block;background:#eff6ff;border:1px solid #93c5fd;border-radius:8px;padding:7px 14px;font-size:13px;margin-right:8px;">
<b style="color:#1e3a8a;">➡ Produces</b>&nbsp; <code>artifacts/02_preprocessing.pkl</code> — read by <code>03_modeling.ipynb</code> and <code>05_visualization.ipynb</code>
</span>
</div>

## <span style="color:#fff;">Load Artifacts</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [ ]:
import numpy as np
import pandas as pd
import os
import pickle

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Projects/DSL/News Classification/'
ARTIFACT_DIR = BASE_PATH + 'artifacts/'

with open(ARTIFACT_DIR + '01_load_data.pkl', 'rb') as f:
    _art01 = pickle.load(f)

train      = _art01['train']
evaluation = _art01['evaluation']
submission = _art01['submission']
SEED       = _art01['SEED']

print(f"✓ Loaded load-data artifacts from {ARTIFACT_DIR}01_load_data.pkl")
print(f"  train {train.shape}, evaluation {evaluation.shape}, submission {submission.shape}")

## <span style="color:#fff;">Explore Data</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

A first look at column types and non-null counts.

In [ ]:
train.info()

## <span style="color:#fff;">Convert Timestamp</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Parses <code>timestamp</code> into a real datetime on both train and evaluation.

In [ ]:
train['timestamp'] = pd.to_datetime(train['timestamp'], errors='coerce')
evaluation['timestamp'] = pd.to_datetime(evaluation['timestamp'], errors='coerce')

print("✓ timestamp parsed to datetime on train and evaluation")

## <span style="color:#fff;">EDA: Class Balance</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Checked before anything else — decides how hard imbalance needs to be fought later (<code>class_weight='balanced'</code>, stratified split, Macro F1 as the metric).

In [ ]:
label_dist = train['label'].value_counts(normalize=True).sort_index()
print("Class balance (label proportions):")
print(label_dist.to_string())
print(f"\nMost frequent class share: {label_dist.max()*100:.1f}%  —  rarest class share: {label_dist.min()*100:.1f}%")

## <span style="color:#fff;">EDA: page_rank Cardinality &amp; Skew</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Decides whether <code>page_rank</code> should be one-hot encoded (few categories) or treated as a scaled numeric feature (many/continuous values).

In [ ]:
from scipy.stats import skew as _skew

print(f"page_rank unique values : {train['page_rank'].nunique()}")
print(train['page_rank'].describe())
try:
    pr_skew = _skew(train['page_rank'].dropna().astype(float))
    print(f"page_rank skewness      : {pr_skew:.3f}")
except Exception as e:
    print("page_rank is not purely numeric — treating as categorical:", e)

# Decide encoding strategy: few unique values -> categorical (one-hot),
# many unique values -> treat as a numeric, scaled feature instead.
PAGE_RANK_AS_NUMERIC = train['page_rank'].nunique() > 20
print(f"\n✓ page_rank will be treated as {'NUMERIC (scaled)' if PAGE_RANK_AS_NUMERIC else 'CATEGORICAL (one-hot)'}")

## <span style="color:#fff;">EDA: source Cardinality</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

High cardinality here is exactly why <code>source</code> gets a target-encoding below, not just a one-hot.

In [ ]:
print(f"source unique values : {train['source'].nunique()}")
print(train['source'].value_counts(normalize=True).head(15))
print("\n✓ used below to decide between one-hot and target-encoding for `source`")

## <span style="color:#fff;">Drop Unused Columns</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

<code>Id</code> is dropped from <code>train</code> only — the evaluation set keeps it, since the final submission has to map predictions back to it.

In [ ]:
# Only drop Id from train — evaluation keeps it, since the final
# submission.csv must map predictions back to the original evaluation Id.
train = train.drop(columns=["Id"], errors="ignore")

train.info()

## <span style="color:#fff;">Train / Validation Split</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Stratified on <code>label</code> so the split itself doesn't distort class balance.

In [ ]:
from sklearn.model_selection import train_test_split

Y = train['label']
X = train.drop(columns=['label'])

X_Train, X_Validation, Y_Train, Y_Validation = train_test_split(
    X, Y, test_size=0.3, random_state=SEED, stratify=Y)

print(f"✓ Stratified train/validation split — X_Train {X_Train.shape}, X_Validation {X_Validation.shape}")

## <span style="color:#fff;">Check Nulls per Feature</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [ ]:
X_Train.isnull().sum()

## <span style="color:#fff;">Handle Nulls</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Train/validation rows with missing labels or malformed nulls are dropped. <b>Evaluation is filled, not dropped</b> — every evaluation row must receive a prediction for the submission to match <code>sample_submission.csv</code>.

In [ ]:
# ── Train / validation: drop rows with a missing label or missing text ──
mask = Y_Train.notna()
X_Train = X_Train[mask]
Y_Train = Y_Train[mask]

mask = Y_Validation.notna()
X_Validation = X_Validation[mask]
Y_Validation = Y_Validation[mask]

X_Train = X_Train.replace(["\\N", "null", "None", "", "nan"], pd.NA)
X_Train = X_Train.replace([np.inf, -np.inf], np.nan)

X_Validation = X_Validation.replace(["\\N", "null", "None", "", "nan"], pd.NA)
X_Validation = X_Validation.replace([np.inf, -np.inf], np.nan)

print("Nulls per feature in X_Train before handling: \n", X_Train.isnull().sum())

X_Train = X_Train.dropna()
X_Validation = X_Validation.dropna()
print("\n Nulls per feature in X_Train after handling: \n", X_Train.isnull().sum())

# ── Evaluation: every row must get a prediction (the submission needs all
# Ids), so fill instead of drop — dropping here would silently shrink the
# evaluation set below what sample_submission.csv expects. ──────────────
evaluation = evaluation.replace(["\\N", "null", "None", "", "nan"], pd.NA)
evaluation = evaluation.replace([np.inf, -np.inf], np.nan)
evaluation['timestamp'] = evaluation['timestamp'].fillna(pd.Timestamp('2020-01-01 12:00:00'))
evaluation = evaluation.fillna({'source': 'unknown', 'title': '', 'article': '', 'page_rank': 0})

print(f"\n✓ evaluation nulls filled (not dropped) — {len(evaluation)} rows preserved for submission")

## <span style="color:#fff;">Handle Duplicate Rows</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Exact <code>title</code>+<code>article</code> duplicates are removed from training only.

In [ ]:
print("Count Duplicated Rows: ", X_Train.duplicated(subset=['title', 'article']).sum())
X_Train = X_Train.drop_duplicates(subset=['title', 'article'])
Y_Train = Y_Train.loc[X_Train.index]
print("Count Duplicates Rows After Dropping: ", X_Train.duplicated(subset=['title', 'article']).sum())

## <span style="color:#fff;">Feature Engineering: source &times; page_rank</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Normalizes text fields. The <code>source</code>/<code>page_rank</code> combination is captured below via K-fold target-encoding instead of a one-hot — see the note there.

In [ ]:
# Convert text features to lowercase
for _df in [X_Train, X_Validation, evaluation]:
    _df['source'] = _df['source'].astype(str).str.lower().str.strip()
    _df['title'] = _df['title'].astype(str).str.lower().str.strip()
    _df['article'] = _df['article'].astype(str).str.lower().str.strip()

print("✓ text fields lowercased/stripped")

## <span style="color:#fff;">Feature Engineering: Text Length &amp; Style</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

<code>article_length</code> was previously computed only for a chart and never fed to the model — fixed here. Adds length (log-scaled, right-skewed), punctuation density, capitalization ratio, and digit density as real model features.

In [ ]:
# ── New: text-length & style features (previously computed only for a
# chart, never fed to the model) ─────────────────────────────────────────
def add_text_style_features(df, title_col='title', article_col='article'):
    raw_title = df[title_col].astype(str)
    raw_article = df[article_col].astype(str)

    df['title_len']   = raw_title.str.split().str.len().fillna(0)
    df['article_len'] = raw_article.str.split().str.len().fillna(0)

    df['title_len_log']   = np.log1p(df['title_len'])
    df['article_len_log'] = np.log1p(df['article_len'])

    df['avg_word_len'] = (raw_article.str.replace(' ', '').str.len() /
                           df['article_len'].replace(0, 1))

    combo = raw_title + ' ' + raw_article
    total_chars = combo.str.len().replace(0, 1)
    df['punct_density'] = combo.str.count(r'[!?]') / total_chars
    df['caps_ratio']    = combo.str.count(r'[A-Z]') / total_chars
    df['digit_density'] = combo.str.count(r'[0-9]') / total_chars
    return df

X_Train      = add_text_style_features(X_Train)
X_Validation = add_text_style_features(X_Validation)
evaluation   = add_text_style_features(evaluation)

NUMERIC_STYLE_COLS = ['title_len_log', 'article_len_log', 'avg_word_len',
                       'punct_density', 'caps_ratio', 'digit_density']
print(f"✓ Text-style features added: {NUMERIC_STYLE_COLS}")
X_Train[NUMERIC_STYLE_COLS].describe()

## <span style="color:#fff;">Feature Engineering: Time Period &amp; Day of Week</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Buckets the publication hour, and adds day-of-week — weekday/weekend publishing patterns often correlate with category (sports on weekends, etc.).

In [ ]:
def get_time_period(hour):
    hour = int(hour)
    if 5 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 15:
        return 'noon'
    elif 15 <= hour < 18:
        return 'afternoon'
    elif 18 <= hour < 21:
        return 'evening'
    elif 21 <= hour < 24:
        return 'night'
    else:
        return 'midnight'

for _df in [X_Train, X_Validation, evaluation]:
    _df['time_period'] = _df['timestamp'].dt.hour.apply(get_time_period)
    _df['day_of_week'] = _df['timestamp'].dt.day_name()

print("✓ time_period bucket and day_of_week extracted from timestamp")

## <span style="color:#fff;">Feature Engineering: source Target-Encoding</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

K-fold multiclass target-encoding of <code>source</code> — one column per class probability. Outlets specialize by topic, so this is expected to be the single biggest accuracy contributor of all the new features.

In [ ]:
# ── New: K-fold multiclass target-encoding for `source` ──────────────────
# Outlets specialize by topic — this usually carries more signal than a
# raw one-hot on (often high-cardinality) source names. One column per
# class = P(label=k | source), computed out-of-fold on train to avoid
# leakage; validation/evaluation use the full-train fit (already held out,
# no leakage risk there).
from sklearn.model_selection import KFold

CLASSES = sorted(Y_Train.unique())
TE_COLS = [f'source_te_{k}' for k in CLASSES]
global_prior = Y_Train.value_counts(normalize=True).reindex(CLASSES).values

def _class_means(sources, labels):
    tmp = pd.DataFrame({'source': sources.values, 'label': labels.values})
    means = tmp.groupby('source')['label'].apply(
        lambda s: pd.Series([ (s == k).mean() for k in CLASSES ], index=TE_COLS)
    )
    return means

# Out-of-fold encoding for TRAIN
oof = pd.DataFrame(index=X_Train.index, columns=TE_COLS, dtype=float)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
for tr_idx, val_idx in kf.split(X_Train):
    fold_means = _class_means(X_Train['source'].iloc[tr_idx], Y_Train.iloc[tr_idx])
    fold_map = X_Train['source'].iloc[val_idx].map(
        lambda s: fold_means.loc[s] if s in fold_means.index else pd.Series(global_prior, index=TE_COLS)
    )
    oof.iloc[val_idx] = pd.DataFrame(list(fold_map.values), index=X_Train.index[val_idx], columns=TE_COLS).values

X_Train[TE_COLS] = oof.astype(float).fillna(pd.Series(global_prior, index=TE_COLS))

# Full-train fit, applied (not re-fit) to validation and evaluation
full_means = _class_means(X_Train['source'], Y_Train)
def _apply_te(df):
    mapped = df['source'].map(lambda s: full_means.loc[s] if s in full_means.index else pd.Series(global_prior, index=TE_COLS))
    return pd.DataFrame(list(mapped.values), index=df.index, columns=TE_COLS).astype(float)

X_Validation[TE_COLS] = _apply_te(X_Validation)
evaluation[TE_COLS]   = _apply_te(evaluation)

print(f"✓ source target-encoded into {len(TE_COLS)} columns (K-fold on train, fit-once on val/eval)")

## <span style="color:#fff;">Text Features: TF-IDF (word + char n-grams)</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Word n-grams for lexical meaning; character n-grams (<code>analyzer='char_wb'</code>) for stylistic/spelling patterns word n-grams miss.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Title is the article's summarization and carries more usable keywords —
# repeat it 3x so it gets proportionally more weight than the longer body.
X_Train_text = ((X_Train['title'] + ' ') * 3 + X_Train['article'])
X_Validation_text = ((X_Validation['title'] + ' ') * 3 + X_Validation['article'])
evaluation_text = ((evaluation['title'] + ' ') * 3 + evaluation['article'])

tfidf_word = TfidfVectorizer(max_features=30000, ngram_range=(1, 3), min_df=3, max_df=0.85, sublinear_tf=True)
X_Train_tfidf = tfidf_word.fit_transform(X_Train_text)
X_Validation_tfidf = tfidf_word.transform(X_Validation_text)
evaluation_tfidf = tfidf_word.transform(evaluation_text)

# ── New: character n-grams — catches stylistic/spelling patterns word
# n-grams miss, and is robust to tokenization noise. ─────────────────────
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=20000,
                              min_df=3, max_df=0.85, sublinear_tf=True)
X_Train_tfidf_char = tfidf_char.fit_transform(X_Train_text)
X_Validation_tfidf_char = tfidf_char.transform(X_Validation_text)
evaluation_tfidf_char = tfidf_char.transform(evaluation_text)

print(f"✓ TF-IDF word: {X_Train_tfidf.shape[1]} features, char: {X_Train_tfidf_char.shape[1]} features")

## <span style="color:#fff;">Feature Selection &amp; Dimensionality Reduction</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Chi-square trims the combined word+char TF-IDF down to the most class-informative terms, then TruncatedSVD (LSA) compresses it into a 500-dim dense, <b>scaled</b> block — this is what makes the text representation GPU-friendly for cuML KNN and the MLP, and scaling it puts it on the same footing as the numeric/target-encoding block it gets combined with.

In [ ]:
from scipy.sparse import hstack
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler as _SVDScaler

X_Train_tfidf_all = hstack([X_Train_tfidf, X_Train_tfidf_char]).tocsr()
X_Validation_tfidf_all = hstack([X_Validation_tfidf, X_Validation_tfidf_char]).tocsr()
evaluation_tfidf_all = hstack([evaluation_tfidf, evaluation_tfidf_char]).tocsr()

# ── Chi-square filter: trims the 50k word+char terms down to the most
# class-informative ones — noise removal, not just speed. ───────────────
K_BEST = min(20000, X_Train_tfidf_all.shape[1])
selector = SelectKBest(chi2, k=K_BEST)
X_Train_tfidf_sel = selector.fit_transform(X_Train_tfidf_all, Y_Train)
X_Validation_tfidf_sel = selector.transform(X_Validation_tfidf_all)
evaluation_tfidf_sel = selector.transform(evaluation_tfidf_all)
print(f"✓ chi2 feature selection: {X_Train_tfidf_all.shape[1]} → {X_Train_tfidf_sel.shape[1]} terms")

# ── TruncatedSVD (LSA): compress into a dense, GPU-friendly block that
# combines cleanly with the numeric/encoded features below, and is what
# every GPU model (cuML KNN/SVM) and the MLP will actually train on. ────
SVD_DIM = 500
svd = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)
X_Train_text_svd = svd.fit_transform(X_Train_tfidf_sel)
X_Validation_text_svd = svd.transform(X_Validation_tfidf_sel)
evaluation_text_svd = svd.transform(evaluation_tfidf_sel)

# ── Scale the SVD output — it was previously combined with the already-
# scaled numeric/target-encoding block without being scaled itself, which
# meant KNN (distance-based) and GaussianNB (variance-based) were seeing
# two feature blocks on very different natural scales. Fit on train only. ──
_svd_scaler = _SVDScaler()
X_Train_text_svd = _svd_scaler.fit_transform(X_Train_text_svd)
X_Validation_text_svd = _svd_scaler.transform(X_Validation_text_svd)
evaluation_text_svd = _svd_scaler.transform(evaluation_text_svd)

print(f"✓ TruncatedSVD: {X_Train_tfidf_sel.shape[1]} → {SVD_DIM} dense components "
      f"(explained variance ≈ {svd.explained_variance_ratio_.sum()*100:.1f}%), scaled")

## <span style="color:#fff;">Encode: page_rank</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Numeric+scaled or one-hot, decided automatically from the EDA cardinality check above.

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

if PAGE_RANK_AS_NUMERIC:
    _pr_scaler = StandardScaler()
    X_Train_pr = _pr_scaler.fit_transform(X_Train[['page_rank']].astype(float))
    X_Validation_pr = _pr_scaler.transform(X_Validation[['page_rank']].astype(float))
    evaluation_pr = _pr_scaler.transform(evaluation[['page_rank']].astype(float))
else:
    enc_pr = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_Train_pr = enc_pr.fit_transform(X_Train[['page_rank']])
    X_Validation_pr = enc_pr.transform(X_Validation[['page_rank']])
    evaluation_pr = enc_pr.transform(evaluation[['page_rank']])

print(f"✓ page_rank encoded — {X_Train_pr.shape[1]} column(s)")

## <span style="color:#fff;">Encode: time_period &amp; day_of_week</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [ ]:
enc_time = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_Train_time = enc_time.fit_transform(X_Train[['time_period']])
X_Validation_time = enc_time.transform(X_Validation[['time_period']])
evaluation_time = enc_time.transform(evaluation[['time_period']])

enc_dow = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_Train_dow = enc_dow.fit_transform(X_Train[['day_of_week']])
X_Validation_dow = enc_dow.transform(X_Validation[['day_of_week']])
evaluation_dow = enc_dow.transform(evaluation[['day_of_week']])

print(f"✓ time_period ({X_Train_time.shape[1]}) and day_of_week ({X_Train_dow.shape[1]}) one-hot encoded")

## <span style="color:#fff;">Combine All Features</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>

Two matrices come out of this: <code>_final</code> (SVD text + numeric + encodings — used by every classical model) and <code>_text_svd</code> (TF-IDF-only — used by the MLP).

In [ ]:
_num_scaler = StandardScaler()
X_Train_numeric = _num_scaler.fit_transform(X_Train[NUMERIC_STYLE_COLS + TE_COLS].values)
X_Validation_numeric = _num_scaler.transform(X_Validation[NUMERIC_STYLE_COLS + TE_COLS].values)
evaluation_numeric = _num_scaler.transform(evaluation[NUMERIC_STYLE_COLS + TE_COLS].values)

# `_final`: everything combined — dense, GPU-ready — used by every
# classical model in 03_modeling.ipynb. (The old source_pagerank one-hot
# block is gone — it was redundant with the source target-encoding above,
# and was most of this matrix's ~1,095 columns, which is a big part of why
# the tree models overfit so badly.)
X_Train_final = np.hstack([
    X_Train_text_svd, X_Train_numeric, X_Train_pr, X_Train_time, X_Train_dow
]).astype(np.float32)
X_Validation_final = np.hstack([
    X_Validation_text_svd, X_Validation_numeric, X_Validation_pr, X_Validation_time, X_Validation_dow
]).astype(np.float32)
evaluation_final = np.hstack([
    evaluation_text_svd, evaluation_numeric, evaluation_pr, evaluation_time, evaluation_dow
]).astype(np.float32)

# `_text_svd`: TF-IDF-only dense block (already scaled) — used only by the
# MLP, per spec.
X_Train_text_svd = X_Train_text_svd.astype(np.float32)
X_Validation_text_svd = X_Validation_text_svd.astype(np.float32)
evaluation_text_svd = evaluation_text_svd.astype(np.float32)

# Realign labels to the surviving index of X_Train / X_Validation — earlier
# steps (null-handling, dedup) drop rows from the feature frames without
# always dropping the matching label row in the same statement.
Y_Train_aligned = Y_Train.loc[X_Train.index].values
Y_Validation_aligned = Y_Validation.loc[X_Validation.index].values

print(f"✓ Combined feature matrix — X_Train_final {X_Train_final.shape} (dense, float32)")
print(f"✓ MLP text-only matrix   — X_Train_text_svd {X_Train_text_svd.shape}")

## <span style="color:#fff;">Save Artifacts</span>
<div style="height:3px;width:48px;background:#2563eb;border-radius:2px;margin:2px 0 10px;"></div>



In [ ]:
with open(ARTIFACT_DIR + '02_preprocessing.pkl', 'wb') as f:
    pickle.dump({
        'X_Train_final': X_Train_final,
        'X_Validation_final': X_Validation_final,
        'evaluation_final': evaluation_final,
        'X_Train_text_svd': X_Train_text_svd,
        'X_Validation_text_svd': X_Validation_text_svd,
        'evaluation_text_svd': evaluation_text_svd,
        'Y_Train_aligned': Y_Train_aligned,
        'Y_Validation_aligned': Y_Validation_aligned,
        'X_Train': X_Train,
        'label_dist': label_dist,
        'evaluation': evaluation,
        'submission': submission,
        'BASE_PATH': BASE_PATH,
        'SEED': SEED,
    }, f)

print(f"✓ Saved preprocessing artifacts to {ARTIFACT_DIR}02_preprocessing.pkl")